# MySQLをベクトルデータベースとして使用し、レコードを登録するサンプル

## パッケージをインポート

In [ ]:
from mysql import connector
from sentence_transformers import SentenceTransformer

## 埋め込みモデル初期化

In [ ]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
model

## MySQLへ接続

In [ ]:
import os
db_config = {
    'user': os.environ['MYSQL_USER'],
    'password': os.environ['MYSQL_PASSWORD'],
    'host': 'llm-rag-examples-mysql',
    'database': os.environ['MYSQL_DATABASE'],
}

db_config

In [ ]:
conn = connector.connect(**db_config)
cursor = conn.cursor()

## テーブル作成（存在しない場合）

In [ ]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS text_embeddings (
        id INT AUTO_INCREMENT PRIMARY KEY,
        text TEXT NOT NULL,
        embedding VECTOR(384) NOT NULL
    )
""")
conn.commit()

## テーブルのレコードを削除

In [ ]:
cursor.execute('TRUNCATE TABLE text_embeddings')
conn.commit()

## 登録されたレコードの確認

In [ ]:
cursor.execute('SELECT id, text FROM text_embeddings ORDER BY id')
records = cursor.fetchall()
for record in records:
    print(f'ID: {record[0]}, Text: {record[1]}')

## ドキュメントをテーブルにレコードとして登録

In [ ]:
texts = [
    '猫は可愛い動物です。',
    '犬は人間の親友と呼ばれています。',
    '東京は日本の首都です。'
]

In [ ]:
import pprint
for text in texts:
    embedding = model.encode([text])[0]
    embedding_str = ','.join(map(str, embedding))
    pprint.pprint(f'[{embedding_str}]')
    print(type(f'[{embedding_str}]'))
    cursor.execute('INSERT INTO text_embeddings (text, embedding) VALUES (%s, STRING_TO_VECTOR(%s))', (text, f'[{embedding_str}]'))

In [ ]:
conn.commit()

In [ ]:
cursor.close()
conn.close()